# Exploración del Modelo 4: costos detallados sala–sala

En este notebook se documenta la implementación y validación del Modelo 4 del problema de asignación de salas con movilidad entre bloques consecutivos.

El Modelo 4 extiende el Modelo 3 incorporando una matriz completa de costos sala–sala. A diferencia del Modelo 3, donde el costo dependía del tipo de transición —misma sala, mismo edificio o distinto edificio—, en esta versión el costo se define directamente para cada par ordenado de salas.

Además, se incorpora explícitamente la posibilidad de que existan:

- estudiantes que salen después del primer bloque;
- estudiantes que entran desde fuera al segundo bloque.

Estos estudiantes no generan costo de movilidad entre salas, porque no realizan una transición sala–sala entre ambos bloques. Sin embargo, sí afectan las restricciones de capacidad de las salas.

## Objetivo del notebook

Este notebook tiene como objetivo verificar que la implementación computacional del Modelo 4 funciona correctamente sobre una batería de tests sintéticos.

En particular, se busca comprobar que:

1. La matriz completa de costos sala–sala se lee correctamente.
2. El modelo utiliza el costo detallado $c_{rs}$ y no una regla binaria o por edificio.
3. Las restricciones de capacidad siguen funcionando en ambos bloques.
4. Los estudiantes salientes afectan la capacidad del bloque 1, pero no generan costo.
5. Los estudiantes entrantes afectan la capacidad del bloque 2, pero no generan costo.
6. El modelo detecta instancias infactibles cuando corresponde.
7. La validación detecta errores estructurales, como una matriz de costos incompleta.
8. Los resultados obtenidos coinciden con los resultados esperados definidos para cada test.

## Descripción breve del Modelo 4

El Modelo 4 mantiene las variables de decisión de los modelos anteriores:

$$
x_{ir} =
\begin{cases}
1, & \text{si el curso } i \text{ del bloque 1 se asigna a la sala } r,\\
0, & \text{en otro caso.}
\end{cases}
$$

$$
y_{ks} =
\begin{cases}
1, & \text{si el curso } k \text{ del bloque 2 se asigna a la sala } s,\\
0, & \text{en otro caso.}
\end{cases}
$$

$$
z_{ikrs} =
\begin{cases}
1, & \text{si el curso } i \text{ está en } r \text{ y el curso } k \text{ está en } s,\\
0, & \text{en otro caso.}
\end{cases}
$$

La función objetivo es:

$$
\min \sum_{i \in I} \sum_{k \in K} \sum_{r \in R} \sum_{s \in R}
f_{ik} c_{rs} z_{ikrs}
$$

donde $c_{rs}$ corresponde al costo detallado de desplazarse desde la sala $r$ hacia la sala $s$.

## Estudiantes salientes y estudiantes entrantes

Para incorporar estudiantes que salen después del primer bloque, se usa el parámetro:

$$
l_i
$$
que representa la cantidad de estudiantes del curso $i$ del bloque 1 que no continúan a ningún curso del bloque 2.

La conservación de estudiantes en el bloque 1 queda dada por:

$$
\sum_{k \in K} f_{ik} + l_i = n_i
$$
Para incorporar estudiantes que entran desde fuera al segundo bloque, se usa el parámetro:

$$
a_k
$$

que representa la cantidad de estudiantes del curso $k$ del bloque 2 que no provienen de ningún curso del bloque 1.

La conservación de estudiantes en el bloque 2 queda dada por:

$$
\sum_{i \in I} f_{ik} + a_k = m_k
$$

Los estudiantes salientes $l_i$ y entrantes $a_k$no aparecen en la función objetivo, pero sí afectan las restricciones de capacidad mediante los tamaños totales $n_i$ y $m_k$.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

from IPython.display import display, Markdown

# Detectar carpeta principal del proyecto
ROOT = Path.cwd()

# Si estamos dentro de notebooks, subimos un nivel
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
DATA_MODELO_4 = ROOT / "data" / "modelo4"

# Agregar src al path
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("Carpeta principal:", ROOT)
print("Carpeta src:", SRC)
print("Carpeta data/modelo4:", DATA_MODELO_4)

Carpeta principal: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II
Carpeta src: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\src
Carpeta data/modelo4: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4


In [ ]:
from generar_data_modelo_4 import generar_todas_las_instancias_modelo_4
from validar_data_4 import validar_instancia_modelo_4, validar_todas_las_instancias
from modelo_4_gurobi import resolver_modelo_4


## Generación de datos sintéticos

Primero se generan las instancias sintéticas del Modelo 4. Cada instancia está diseñada para probar una propiedad específica del modelo.

In [ ]:
generar_todas_las_instancias_modelo_4()

Instancia creada: C:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4\M4_T01_misma_sala_costo_0
Instancia creada: C:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4\M4_T02_costo_detallado_forzado
Instancia creada: C:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4\M4_T03_eleccion_por_matriz_costos
Instancia creada: C:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4\M4_T04_estudiantes_entrantes
Instancia creada: C:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4\M4_T05_estudiantes_salientes
Instancia creada: C:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4\M4_T06_entrantes_salientes_cambian_optimo
Instancia creada: C:\Users\yuvi8\Desktop\Semestre Actual\La

## Tests disponibles

A continuación se revisan las carpetas de tests generadas para el Modelo 4.

In [ ]:
carpetas_tests = sorted([
    carpeta for carpeta in DATA_MODELO_4.iterdir()
    if carpeta.is_dir()
])

for carpeta in carpetas_tests:
    print(carpeta.name)

M4_T01_misma_sala_costo_0
M4_T02_costo_detallado_forzado
M4_T03_eleccion_por_matriz_costos
M4_T04_estudiantes_entrantes
M4_T05_estudiantes_salientes
M4_T06_entrantes_salientes_cambian_optimo
M4_T07_infactible_por_entrantes
M4_T08_infactible_por_salientes
M4_T09_matriz_costos_incompleta
M4_T10_costos_no_equivalentes_a_gamma


## Resumen de tests

Cada test contiene un archivo `metadata.json`, donde se registra el propósito de la instancia, el status esperado y el costo esperado cuando corresponde.

In [ ]:
resumen_tests = []

for carpeta in carpetas_tests:
    with open(carpeta / "metadata.json", "r", encoding="utf-8") as archivo:
        metadata = json.load(archivo)

    resumen_tests.append({
        "test": carpeta.name,
        "proposito": metadata.get("proposito"),
        "status_esperado": metadata.get("status_esperado"),
        "costo_esperado": metadata.get("costo_esperado")
    })

tabla_resumen_tests = pd.DataFrame(resumen_tests)
tabla_resumen_tests

,test,proposito,status_esperado,costo_esperado
0,M4_T01_misma_sala_costo_0,Verificar que permanecer en la misma sala tien...,OPTIMAL,0.0
1,M4_T02_costo_detallado_forzado,Verificar que se utiliza el costo detallado sa...,OPTIMAL,120.0
2,M4_T03_eleccion_por_matriz_costos,Verificar que el modelo escoge las salas de me...,OPTIMAL,8.0
3,M4_T04_estudiantes_entrantes,Verificar que estudiantes entrantes aumentan e...,OPTIMAL,0.0
4,M4_T05_estudiantes_salientes,Verificar que estudiantes salientes afectan la...,OPTIMAL,0.0
5,M4_T06_entrantes_salientes_cambian_optimo,Verificar que entrantes y salientes no generan...,OPTIMAL,120.0
6,M4_T07_infactible_por_entrantes,Verificar infactibilidad cuando estudiantes en...,INFEASIBLE,NaN
7,M4_T08_infactible_por_salientes,Verificar infactibilidad cuando estudiantes sa...,INFEASIBLE,NaN
8,M4_T09_matriz_costos_incompleta,Dejar una matriz de costos incompleta para pro...,ERROR_VALIDACION,NaN
9,M4_T10_costos_no_equivalentes_a_gamma,Verificar que el Modelo 4 usa directamente la ...,OPTIMAL,16.0


## Validación de datos

Antes de resolver cada instancia con Gurobi, se valida que los datos estén correctamente construidos.

La validación revisa:

- existencia de todos los archivos requeridos;
- columnas obligatorias;
- ausencia de valores negativos;
- consistencia entre cursos, salas y flujos;
- conservación de estudiantes en ambos bloques;
- presencia de estudiantes salientes y entrantes;
- matriz completa de costos sala–sala;
- diagonal de costos igual a cero.

El test `M4_T09_matriz_costos_incompleta` fue diseñado intencionalmente para fallar en la validación.

In [ ]:
validar_todas_las_instancias(DATA_MODELO_4)

Validando instancias en: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4

[OK] M4_T01_misma_sala_costo_0
[OK] M4_T02_costo_detallado_forzado
[OK] M4_T03_eleccion_por_matriz_costos
[OK] M4_T04_estudiantes_entrantes
[OK] M4_T05_estudiantes_salientes
[OK] M4_T06_entrantes_salientes_cambian_optimo
[OK] M4_T07_infactible_por_entrantes
     [ADVERTENCIA] Curso B2 k1 tiene tamano 60 mayor que la capacidad máxima disponible 50.
[OK] M4_T08_infactible_por_salientes
     [ADVERTENCIA] Curso B1 i1 tiene tamano 60 mayor que la capacidad máxima disponible 50.
[OK] M4_T09_matriz_costos_incompleta
     Error de validación esperado: La matriz de costos está incompleta. Faltan pares: [('r3', 'r1')]
[OK] M4_T10_costos_no_equivalentes_a_gamma

Resumen
Total de instancias revisadas: 10
Instancias válidas: 9
Errores de validación esperados: 1
Errores no esperados: 0


## Funciones auxiliares para mostrar datos y resultados

In [ ]:
from IPython.display import display, Markdown
import pandas as pd
import json


def leer_csv_test(nombre_test, archivo):
    return pd.read_csv(DATA_MODELO_4 / nombre_test / archivo)


def leer_metadata_test(nombre_test):
    with open(DATA_MODELO_4 / nombre_test / "metadata.json", "r", encoding="utf-8") as archivo:
        return json.load(archivo)


def mostrar_flujos(nombre_test):
    flujos = leer_csv_test(nombre_test, "flujos.csv")
    display(Markdown("**Flujos entre cursos que sí generan movilidad:**"))
    display(flujos)


def mostrar_libres_entrantes(nombre_test):
    libres = leer_csv_test(nombre_test, "libres.csv")
    entrantes = leer_csv_test(nombre_test, "entrantes.csv")

    display(Markdown("**Estudiantes salientes desde el bloque 1:**"))
    display(libres)

    display(Markdown("**Estudiantes entrantes al bloque 2:**"))
    display(entrantes)


def mostrar_matriz_costos(nombre_test):
    costos = leer_csv_test(nombre_test, "costos_sala_sala.csv")

    display(Markdown("**Matriz de costos sala–sala:**"))

    try:
        matriz = costos.pivot(
            index="sala_origen",
            columns="sala_destino",
            values="costo"
        )
        display(matriz)
    except Exception:
        display(costos)


def resolver_y_resumir(nombre_test):
    metadata = leer_metadata_test(nombre_test)

    status_esperado = metadata.get("status_esperado")
    costo_esperado = metadata.get("costo_esperado")

    try:
        resultado = resolver_modelo_4(
            DATA_MODELO_4 / nombre_test,
            mostrar_output=False,
            validar=True
        )

        status_obtenido = resultado["status"]
        costo_obtenido = resultado["objetivo"]

        if costo_esperado is None or costo_obtenido is None:
            coincide_costo = "No aplica"
        else:
            coincide_costo = abs(float(costo_obtenido) - float(costo_esperado)) <= 1e-6

        resumen = pd.DataFrame([{
            "status_esperado": status_esperado,
            "status_obtenido": status_obtenido,
            "costo_esperado": costo_esperado,
            "costo_obtenido": costo_obtenido,
            "cumple_status": status_obtenido == status_esperado,
            "cumple_costo": coincide_costo
        }])

        display(Markdown("**Resultado del test:**"))
        display(resumen)

        return resultado

    except Exception as e:
        resumen = pd.DataFrame([{
            "status_esperado": status_esperado,
            "status_obtenido": "ERROR_VALIDACION",
            "costo_esperado": costo_esperado,
            "costo_obtenido": None,
            "cumple_status": status_esperado == "ERROR_VALIDACION",
            "cumple_costo": "No aplica",
            "mensaje": str(e)
        }])

        display(Markdown("**Resultado del test:**"))
        display(resumen)

        return None


def mostrar_asignacion_compacta(resultado):
    if resultado is None or resultado["status"] != "OPTIMAL":
        return

    columnas = ["sala", "curso_b1", "curso_b2"]
    display(Markdown("**Asignación obtenida por sala:**"))
    display(resultado["df_salas"][columnas])


def mostrar_movimientos_compactos(resultado):
    if resultado is None or resultado["status"] != "OPTIMAL":
        return

    movimientos = resultado["df_movimientos"].copy()
    movimientos["misma_sala"] = movimientos["sala_origen"] == movimientos["sala_destino"]

    columnas = [
        "curso_b1",
        "sala_origen",
        "curso_b2",
        "sala_destino",
        "flujo",
        "costo_unitario",
        "costo_total",
        "misma_sala"
    ]

    display(Markdown("**Movimientos que determinan el costo:**"))
    display(movimientos[columnas])

def mostrar_salas_y_edificios(nombre_test):
    salas = leer_csv_test(nombre_test, "salas.csv")

    columnas = ["sala_id", "capacidad"]

    if "edificio" in salas.columns:
        columnas.append("edificio")

    display(Markdown("**Salas, capacidades y edificios:**"))
    display(salas[columnas])

def mostrar_capacidades_relevantes(nombre_test):
    cursos_b1 = leer_csv_test(nombre_test, "cursos_b1.csv")
    cursos_b2 = leer_csv_test(nombre_test, "cursos_b2.csv")
    salas = leer_csv_test(nombre_test, "salas.csv")

    display(Markdown("**Cursos del bloque 1:**"))
    display(cursos_b1)

    display(Markdown("**Cursos del bloque 2:**"))
    display(cursos_b2)

    display(Markdown("**Capacidades disponibles:**"))
    display(salas[["sala_id", "capacidad"]])

## M4_T01 — Misma sala, costo 0

Este test está diseñado para verificar el caso más básico del Modelo 4: cuando los flujos entre cursos son directos y las capacidades lo permiten, todos los estudiantes que continúan al segundo bloque deberían permanecer en la misma sala.

Se tienen dos flujos principales: $i1 \rightarrow k1$ y $i2 \rightarrow k2$. No hay estudiantes entrantes ni salientes.

Lo que se busca probar es que permanecer en la misma sala genera costo cero y que el solver efectivamente encuentra una asignación sin movilidad.

Resultado esperado: `OPTIMAL`, costo `0`.

In [ ]:
nombre_test = "M4_T01_misma_sala_costo_0"

mostrar_flujos(nombre_test)

resultado = resolver_y_resumir(nombre_test)

mostrar_asignacion_compacta(resultado)
mostrar_movimientos_compactos(resultado)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,30
1,i2,k2,20


Set parameter Username
Set parameter LicenseID to value 2808412
Academic license - for non-commercial use only - expires 2027-04-16


**Resultado del test:**

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,OPTIMAL,OPTIMAL,0,0.0,True,True


**Asignación obtenida por sala:**

,sala,curso_b1,curso_b2
0,r1,i1,k1
1,r2,i2,k2


**Movimientos que determinan el costo:**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total,misma_sala
0,i1,r1,k1,r1,30.0,0.0,0.0,True
1,i2,r2,k2,r2,20.0,0.0,0.0,True


## M4_T02 — Costo detallado forzado por capacidad

Este test verifica que el Modelo 4 usa el costo detallado sala–sala y no un costo binario.

Las capacidades fuerzan que $i1$ y $k2$ ocupen la sala grande, mientras que $i2$ y $k1$ deben ocupar la sala pequeña. Por lo tanto, algunos flujos deben cruzar entre salas. Como el costo entre $r1$ y $r2$ es 2, el costo total esperado es 120.

Resultado esperado: `OPTIMAL`, costo `120`.

In [ ]:
nombre_test = "M4_T02_costo_detallado_forzado"

mostrar_flujos(nombre_test)
mostrar_matriz_costos(nombre_test)

resultado = resolver_y_resumir(nombre_test)

mostrar_asignacion_compacta(resultado)
mostrar_movimientos_compactos(resultado)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,30
1,i1,k2,20
2,i2,k2,30


**Matriz de costos sala–sala:**

sala_destino,r1,r2
sala_origen,,
r1,0,2
r2,2,0


**Resultado del test:**

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,OPTIMAL,OPTIMAL,120,120.0,True,True


**Asignación obtenida por sala:**

,sala,curso_b1,curso_b2
0,r1,i1,k2
1,r2,i2,k1


**Movimientos que determinan el costo:**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total,misma_sala
0,i1,r1,k1,r2,30.0,2.0,60.0,False
1,i1,r1,k2,r1,20.0,0.0,0.0,True
2,i2,r2,k2,r1,30.0,2.0,60.0,False


## M4_T03 — Elección de salas por matriz de costos

Este test verifica que el modelo elige las salas más convenientes según la matriz de costos.

Existen tres salas disponibles, pero solo dos cursos por bloque. La matriz de costos hace que usar $r1$ y $r2$ sea más barato que usar combinaciones con $r3$ Por eso, el modelo debería preferir las salas de menor costo relativo.

Resultado esperado: `OPTIMAL`, costo `8`.

In [ ]:
nombre_test = "M4_T03_eleccion_por_matriz_costos"

mostrar_flujos(nombre_test)
mostrar_matriz_costos(nombre_test)

resultado = resolver_y_resumir(nombre_test)

mostrar_asignacion_compacta(resultado)
mostrar_movimientos_compactos(resultado)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,6
1,i1,k2,4
2,i2,k1,4
3,i2,k2,6


**Matriz de costos sala–sala:**

sala_destino,r1,r2,r3
sala_origen,,,
r1,0,1,5
r2,1,0,5
r3,5,5,0


**Resultado del test:**

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,OPTIMAL,OPTIMAL,8,8.0,True,True


**Asignación obtenida por sala:**

,sala,curso_b1,curso_b2
0,r1,i2,k2
1,r2,i1,k1
2,r3,—,—


**Movimientos que determinan el costo:**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total,misma_sala
0,i1,r2,k1,r2,6.0,0.0,0.0,True
1,i1,r2,k2,r1,4.0,1.0,4.0,False
2,i2,r1,k1,r2,4.0,1.0,4.0,False
3,i2,r1,k2,r1,6.0,0.0,0.0,True


## M4_T04 — Estudiantes entrantes al segundo bloque

Este test verifica el tratamiento de estudiantes que llegan desde fuera al bloque 2.

El curso $k1$ recibe estudiantes desde $i1$ pero además recibe estudiantes entrantes. Estos estudiantes aumentan el tamaño total de $k1$ por lo que afectan la capacidad requerida. Sin embargo, no generan costo de movilidad, porque no tienen sala de origen en el bloque 1.

Lo que se busca comprobar es que el costo siga siendo cero si los estudiantes que sí continúan entre bloques pueden quedarse en la misma sala.

Resultado esperado: `OPTIMAL`, costo `0`.

In [ ]:
nombre_test = "M4_T04_estudiantes_entrantes"

mostrar_flujos(nombre_test)
mostrar_libres_entrantes(nombre_test)

resultado = resolver_y_resumir(nombre_test)

mostrar_asignacion_compacta(resultado)
mostrar_movimientos_compactos(resultado)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,30
1,i2,k2,20


**Estudiantes salientes desde el bloque 1:**

,curso_b1,libres
0,i1,0
1,i2,0


**Estudiantes entrantes al bloque 2:**

...

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,OPTIMAL,OPTIMAL,0,0.0,True,True


**Asignación obtenida por sala:**

,sala,curso_b1,curso_b2
0,r1,i1,k1
1,r2,i2,k2
2,r3,—,—


**Movimientos que determinan el costo:**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total,misma_sala
0,i1,r1,k1,r1,30.0,0.0,0.0,True
1,i2,r2,k2,r2,20.0,0.0,0.0,True


## M4_T05 — Estudiantes salientes desde el primer bloque

Este test verifica el tratamiento de estudiantes que tuvieron clase en el bloque 1, pero no continúan al bloque 2.

El curso $i1$ tiene estudiantes salientes. Estos estudiantes cuentan para la capacidad del bloque 1, pero no generan costo de movilidad porque no tienen sala de destino en el bloque 2.

Lo que se busca comprobar es que el modelo respeta la capacidad total del curso en el bloque 1 y que solo cobra costo por los estudiantes que efectivamente continúan hacia un curso del bloque 2.

Resultado esperado: `OPTIMAL`, costo `0`.

In [ ]:
nombre_test = "M4_T05_estudiantes_salientes"

mostrar_flujos(nombre_test)
mostrar_libres_entrantes(nombre_test)

resultado = resolver_y_resumir(nombre_test)

mostrar_asignacion_compacta(resultado)
mostrar_movimientos_compactos(resultado)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,30
1,i2,k2,20


**Estudiantes salientes desde el bloque 1:**

,curso_b1,libres
0,i1,20
1,i2,0


**Estudiantes entrantes al bloque 2:**

...

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,OPTIMAL,OPTIMAL,0,0.0,True,True


**Asignación obtenida por sala:**

,sala,curso_b1,curso_b2
0,r1,i1,k1
1,r2,i2,k2
2,r3,—,—


**Movimientos que determinan el costo:**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total,misma_sala
0,i1,r1,k1,r1,30.0,0.0,0.0,True
1,i2,r2,k2,r2,20.0,0.0,0.0,True


## M4_T06 — Entrantes y salientes cambian el óptimo por capacidad

Este test combina estudiantes entrantes y salientes.

Los estudiantes salientes no generan costo después del bloque 1, y los entrantes no generan costo desde el bloque 1 hacia el bloque 2. Sin embargo, ambos afectan los tamaños totales de los cursos y, por tanto, las restricciones de capacidad.

En esta instancia, esas restricciones fuerzan una asignación donde los flujos que sí continúan deben cambiar de sala. El objetivo del test es comprobar que los entrantes y salientes no aparecen directamente en la función objetivo, pero sí pueden cambiar la solución óptima mediante capacidad.

Resultado esperado: `OPTIMAL`, costo `120`.

In [ ]:
nombre_test = "M4_T06_entrantes_salientes_cambian_optimo"

mostrar_flujos(nombre_test)
mostrar_libres_entrantes(nombre_test)
mostrar_matriz_costos(nombre_test)

resultado = resolver_y_resumir(nombre_test)

mostrar_asignacion_compacta(resultado)
mostrar_movimientos_compactos(resultado)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,30
1,i2,k2,30


**Estudiantes salientes desde el bloque 1:**

,curso_b1,libres
0,i1,20
1,i2,0


**Estudiantes entrantes al bloque 2:**

...

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,OPTIMAL,OPTIMAL,120,120.0,True,True


**Asignación obtenida por sala:**

,sala,curso_b1,curso_b2
0,r1,i1,k2
1,r2,i2,k1


**Movimientos que determinan el costo:**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total,misma_sala
0,i1,r1,k1,r2,30.0,2.0,60.0,False
1,i2,r2,k2,r1,30.0,2.0,60.0,False


## M4_T07 — Infactible por estudiantes entrantes

Este test verifica que los estudiantes entrantes afectan la capacidad requerida en el bloque 2.

El curso $k1$ recibe estudiantes desde el bloque 1 y además estudiantes entrantes desde fuera. Su tamaño total supera la capacidad máxima disponible, por lo que no existe ninguna sala capaz de alojarlo.

Lo que se busca probar es que el modelo detecta correctamente la infactibilidad causada por estudiantes que llegan al segundo bloque.

Resultado esperado: `INFEASIBLE`.

In [ ]:
nombre_test = "M4_T07_infactible_por_entrantes"

mostrar_flujos(nombre_test)
mostrar_libres_entrantes(nombre_test)
mostrar_capacidades_relevantes(nombre_test)

resultado = resolver_y_resumir(nombre_test)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,30


**Estudiantes salientes desde el bloque 1:**

,curso_b1,libres
0,i1,0


**Estudiantes entrantes al bloque 2:**

...

,curso_id,tamano
0,k1,60


**Capacidades disponibles:**

,sala_id,capacidad
0,r1,50
1,r2,30


**Resultado del test:**

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,INFEASIBLE,INFEASIBLE,None,None,True,No aplica


## M4_T08 — Infactible por estudiantes salientes

Este test verifica que los estudiantes salientes afectan la capacidad requerida en el bloque 1.

Aunque parte de los estudiantes de $i1$ no continúan al segundo bloque, todos estuvieron presentes durante el bloque 1. Por lo tanto, el curso $i1$ debe caber completo en alguna sala del bloque 1.

Como el tamaño de $i1$ supera la capacidad máxima disponible, la instancia debe ser infactible.

Resultado esperado: `INFEASIBLE`.

In [ ]:
nombre_test = "M4_T08_infactible_por_salientes"

mostrar_flujos(nombre_test)
mostrar_libres_entrantes(nombre_test)
mostrar_capacidades_relevantes(nombre_test)

resultado = resolver_y_resumir(nombre_test)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,30


**Estudiantes salientes desde el bloque 1:**

,curso_b1,libres
0,i1,30


**Estudiantes entrantes al bloque 2:**

...

,curso_id,tamano
0,k1,30


**Capacidades disponibles:**

,sala_id,capacidad
0,r1,50
1,r2,30


**Resultado del test:**

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,INFEASIBLE,INFEASIBLE,None,None,True,No aplica


## M4_T09 — Matriz de costos incompleta

Este test no está diseñado para ser resuelto por Gurobi.

Su objetivo es verificar que la etapa de validación detecte una matriz de costos sala–sala incompleta. En el Modelo 4 debe existir un costo para cada par ordenado de salas $(r,s)$

Por lo tanto, el resultado esperado no es `OPTIMAL` ni `INFEASIBLE`, sino un error de validación antes de construir el modelo de optimización.

Resultado esperado: `ERROR_VALIDACION`.

In [ ]:
nombre_test = "M4_T09_matriz_costos_incompleta"

mostrar_matriz_costos(nombre_test)

resultado = resolver_y_resumir(nombre_test)

**Matriz de costos sala–sala:**

sala_destino,r1,r2,r3
sala_origen,,,
r1,0.0,1.0,5.0
r2,1.0,0.0,5.0
r3,NaN,5.0,0.0


**Resultado del test:**

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo,mensaje
0,ERROR_VALIDACION,ERROR_VALIDACION,None,None,True,No aplica,La matriz de costos está incompleta. Faltan pa...


## M4_T10 — Costos detallados no equivalentes a una regla por edificio

Este test verifica la diferencia conceptual entre el Modelo 3 y el Modelo 4.

En el Modelo 3, el costo se construía a partir del tipo de transición: misma sala, mismo edificio o distinto edificio. En cambio, en el Modelo 4 el costo se toma directamente desde la matriz sala–sala $c_{rs}$.

La instancia incluye salas en distintos edificios, pero la matriz de costos está diseñada para que una sala de otro edificio sea más barata que una sala del mismo edificio. Por ejemplo, $r1$ y $r2$ pertenecen al edificio A, pero su costo de traslado es mayor que el costo entre $ r1$  y $ r3$ , aun cuando $ r3$ pertenece al edificio B.

Por lo tanto, se busca comprobar que el modelo decide según el costo detallado sala–sala y no según una regla fija basada en edificios.

Resultado esperado: `OPTIMAL`, costo `16`.

In [ ]:
nombre_test = "M4_T10_costos_no_equivalentes_a_gamma"

mostrar_flujos(nombre_test)
mostrar_salas_y_edificios(nombre_test)
mostrar_matriz_costos(nombre_test)

resultado = resolver_y_resumir(nombre_test)

mostrar_asignacion_compacta(resultado)
mostrar_movimientos_compactos(resultado)

**Flujos entre cursos que sí generan movilidad:**

,curso_b1,curso_b2,flujo
0,i1,k1,6
1,i1,k2,4
2,i2,k1,4
3,i2,k2,6


**Salas, capacidades y edificios:**

,sala_id,capacidad,edificio
0,r1,10,A
1,r2,10,A
2,r3,10,B


**Matriz de costos sala–sala:**

...

,status_esperado,status_obtenido,costo_esperado,costo_obtenido,cumple_status,cumple_costo
0,OPTIMAL,OPTIMAL,16,16.0,True,True


**Asignación obtenida por sala:**

,sala,curso_b1,curso_b2
0,r1,i1,k1
1,r2,—,—
2,r3,i2,k2


**Movimientos que determinan el costo:**

,curso_b1,sala_origen,curso_b2,sala_destino,flujo,costo_unitario,costo_total,misma_sala
0,i1,r1,k1,r1,6.0,0.0,0.0,True
1,i1,r1,k2,r3,4.0,2.0,8.0,False
2,i2,r3,k1,r1,4.0,2.0,8.0,False
3,i2,r3,k2,r3,6.0,0.0,0.0,True


## Ejemplo — Campus con 3 edificios y costos detallados sala–sala

En esta sección se construye una instancia más desafiante del Modelo 4. A diferencia de los tests anteriores, que estaban diseñados para verificar propiedades específicas de forma aislada, esta instancia combina varios elementos del problema real:

- 3 edificios del campus: A, B y C.
- 15 salas disponibles, con entre 4 y 6 salas por edificio.
- Capacidades heterogéneas entre salas.
- 12 cursos en el bloque 1 y 12 cursos en el bloque 2.
- Flujos divididos entre cursos, es decir, estudiantes de un curso del bloque 1 pueden continuar hacia más de un curso del bloque 2.
- Estudiantes salientes, que tuvieron clase en el bloque 1 pero no continúan al bloque 2.
- Estudiantes entrantes, que llegan al bloque 2 sin haber tenido clase en el bloque 1.
- Una matriz completa de costos sala–sala construida a partir de una ubicación sintética de cada sala.

El objetivo es que el modelo decida simultáneamente la asignación de salas para ambos bloques, minimizando el costo total de movilidad de los estudiantes que sí transitan entre cursos. Los estudiantes entrantes y salientes no generan costo directo de movilidad, pero sí afectan las restricciones de capacidad.

In [ ]:
from pathlib import Path
import shutil
import json
import math
import pandas as pd


# Carpeta de la instancia 
NOMBRE_DESAFIO = "M4_DESAFIO_3_edificios"
CARPETA_DESAFIO = DATA_MODELO_4 / NOMBRE_DESAFIO

if CARPETA_DESAFIO.exists():
    shutil.rmtree(CARPETA_DESAFIO)

CARPETA_DESAFIO.mkdir(parents=True, exist_ok=True)


def guardar_csv(df, nombre_archivo):
    df.to_csv(CARPETA_DESAFIO / nombre_archivo, index=False)


# ------------------------------------------------------------
# 1. Salas: 3 edificios, 15 salas en total
# ------------------------------------------------------------

salas = pd.DataFrame([
    # Edificio A: 4 salas
    {"sala_id": "A101", "capacidad": 45, "edificio": "A", "piso": 1, "coord_x": 0,  "coord_y": 0},
    {"sala_id": "A102", "capacidad": 40, "edificio": "A", "piso": 1, "coord_x": 6,  "coord_y": 0},
    {"sala_id": "A201", "capacidad": 35, "edificio": "A", "piso": 2, "coord_x": 0,  "coord_y": 6},
    {"sala_id": "A202", "capacidad": 30, "edificio": "A", "piso": 2, "coord_x": 6,  "coord_y": 6},

    # Edificio B: 5 salas
    {"sala_id": "B101", "capacidad": 60, "edificio": "B", "piso": 1, "coord_x": 90, "coord_y": 20},
    {"sala_id": "B102", "capacidad": 50, "edificio": "B", "piso": 1, "coord_x": 98, "coord_y": 20},
    {"sala_id": "B201", "capacidad": 45, "edificio": "B", "piso": 2, "coord_x": 90, "coord_y": 28},
    {"sala_id": "B202", "capacidad": 35, "edificio": "B", "piso": 2, "coord_x": 98, "coord_y": 28},
    {"sala_id": "B301", "capacidad": 30, "edificio": "B", "piso": 3, "coord_x": 94, "coord_y": 24},

    # Edificio C: 6 salas
    {"sala_id": "C101", "capacidad": 55, "edificio": "C", "piso": 1, "coord_x": 35, "coord_y": 95},
    {"sala_id": "C102", "capacidad": 45, "edificio": "C", "piso": 1, "coord_x": 43, "coord_y": 95},
    {"sala_id": "C201", "capacidad": 40, "edificio": "C", "piso": 2, "coord_x": 35, "coord_y": 103},
    {"sala_id": "C202", "capacidad": 35, "edificio": "C", "piso": 2, "coord_x": 43, "coord_y": 103},
    {"sala_id": "C301", "capacidad": 30, "edificio": "C", "piso": 3, "coord_x": 35, "coord_y": 111},
    {"sala_id": "C302", "capacidad": 25, "edificio": "C", "piso": 3, "coord_x": 43, "coord_y": 111},
])


# ------------------------------------------------------------
# 2. Cursos del bloque 1
# ------------------------------------------------------------

cursos_b1 = pd.DataFrame([
    {"curso_id": "i1",  "tamano": 60},
    {"curso_id": "i2",  "tamano": 55},
    {"curso_id": "i3",  "tamano": 50},
    {"curso_id": "i4",  "tamano": 45},
    {"curso_id": "i5",  "tamano": 45},
    {"curso_id": "i6",  "tamano": 40},
    {"curso_id": "i7",  "tamano": 40},
    {"curso_id": "i8",  "tamano": 35},
    {"curso_id": "i9",  "tamano": 35},
    {"curso_id": "i10", "tamano": 30},
    {"curso_id": "i11", "tamano": 30},
    {"curso_id": "i12", "tamano": 25},
])


# ------------------------------------------------------------
# 3. Cursos del bloque 2
# ------------------------------------------------------------

cursos_b2 = pd.DataFrame([
    {"curso_id": "k1",  "tamano": 55},
    {"curso_id": "k2",  "tamano": 50},
    {"curso_id": "k3",  "tamano": 45},
    {"curso_id": "k4",  "tamano": 45},
    {"curso_id": "k5",  "tamano": 40},
    {"curso_id": "k6",  "tamano": 40},
    {"curso_id": "k7",  "tamano": 35},
    {"curso_id": "k8",  "tamano": 35},
    {"curso_id": "k9",  "tamano": 30},
    {"curso_id": "k10", "tamano": 30},
    {"curso_id": "k11", "tamano": 30},
    {"curso_id": "k12", "tamano": 25},
])


# ------------------------------------------------------------
# 4. Flujos entre cursos
# ------------------------------------------------------------

flujos = pd.DataFrame([
    {"curso_b1": "i1",  "curso_b2": "k1",  "flujo": 35},
    {"curso_b1": "i1",  "curso_b2": "k2",  "flujo": 20},

    {"curso_b1": "i2",  "curso_b2": "k2",  "flujo": 30},
    {"curso_b1": "i2",  "curso_b2": "k3",  "flujo": 20},

    {"curso_b1": "i3",  "curso_b2": "k3",  "flujo": 25},
    {"curso_b1": "i3",  "curso_b2": "k4",  "flujo": 20},

    {"curso_b1": "i4",  "curso_b2": "k4",  "flujo": 25},
    {"curso_b1": "i4",  "curso_b2": "k5",  "flujo": 15},

    {"curso_b1": "i5",  "curso_b2": "k5",  "flujo": 25},
    {"curso_b1": "i5",  "curso_b2": "k6",  "flujo": 15},

    {"curso_b1": "i6",  "curso_b2": "k6",  "flujo": 20},
    {"curso_b1": "i6",  "curso_b2": "k7",  "flujo": 15},

    {"curso_b1": "i7",  "curso_b2": "k7",  "flujo": 20},
    {"curso_b1": "i7",  "curso_b2": "k8",  "flujo": 15},

    {"curso_b1": "i8",  "curso_b2": "k8",  "flujo": 20},
    {"curso_b1": "i8",  "curso_b2": "k9",  "flujo": 10},

    {"curso_b1": "i9",  "curso_b2": "k9",  "flujo": 20},
    {"curso_b1": "i9",  "curso_b2": "k10", "flujo": 15},

    {"curso_b1": "i10", "curso_b2": "k10", "flujo": 15},
    {"curso_b1": "i10", "curso_b2": "k11", "flujo": 12},

    {"curso_b1": "i11", "curso_b2": "k11", "flujo": 18},
    {"curso_b1": "i11", "curso_b2": "k12", "flujo": 10},

    {"curso_b1": "i12", "curso_b2": "k12", "flujo": 15},
    {"curso_b1": "i12", "curso_b2": "k1",  "flujo": 10},
])


# ------------------------------------------------------------
# 5. Estudiantes salientes desde bloque 1
# ------------------------------------------------------------

libres = pd.DataFrame([
    {"curso_b1": "i1",  "libres": 5},
    {"curso_b1": "i2",  "libres": 5},
    {"curso_b1": "i3",  "libres": 5},
    {"curso_b1": "i4",  "libres": 5},
    {"curso_b1": "i5",  "libres": 5},
    {"curso_b1": "i6",  "libres": 5},
    {"curso_b1": "i7",  "libres": 5},
    {"curso_b1": "i8",  "libres": 5},
    {"curso_b1": "i9",  "libres": 0},
    {"curso_b1": "i10", "libres": 3},
    {"curso_b1": "i11", "libres": 2},
    {"curso_b1": "i12", "libres": 0},
])


# ------------------------------------------------------------
# 6. Estudiantes entrantes al bloque 2
# ------------------------------------------------------------

entrantes = pd.DataFrame([
    {"curso_b2": "k1",  "entrantes": 10},
    {"curso_b2": "k2",  "entrantes": 0},
    {"curso_b2": "k3",  "entrantes": 0},
    {"curso_b2": "k4",  "entrantes": 0},
    {"curso_b2": "k5",  "entrantes": 0},
    {"curso_b2": "k6",  "entrantes": 5},
    {"curso_b2": "k7",  "entrantes": 0},
    {"curso_b2": "k8",  "entrantes": 0},
    {"curso_b2": "k9",  "entrantes": 0},
    {"curso_b2": "k10", "entrantes": 0},
    {"curso_b2": "k11", "entrantes": 0},
    {"curso_b2": "k12", "entrantes": 0},
])


# ------------------------------------------------------------
# 7. Matriz completa de costos sala-sala
# ------------------------------------------------------------

def costo_entre_salas(sala_origen, sala_destino):
    if sala_origen["sala_id"] == sala_destino["sala_id"]:
        return 0.0

    dx = sala_origen["coord_x"] - sala_destino["coord_x"]
    dy = sala_origen["coord_y"] - sala_destino["coord_y"]
    distancia_planta = math.sqrt(dx**2 + dy**2)
    diferencia_piso = abs(sala_origen["piso"] - sala_destino["piso"])

    mismo_edificio = sala_origen["edificio"] == sala_destino["edificio"]

    if mismo_edificio:
        costo = 1.0 + 0.03 * distancia_planta + 0.70 * diferencia_piso
    else:
        costo = 2.0 + 0.045 * distancia_planta + 0.35 * diferencia_piso

    return round(costo, 2)


filas_costos = []

for _, origen in salas.iterrows():
    for _, destino in salas.iterrows():
        filas_costos.append({
            "sala_origen": origen["sala_id"],
            "sala_destino": destino["sala_id"],
            "costo": costo_entre_salas(origen, destino)
        })

costos_sala_sala = pd.DataFrame(filas_costos)


# ------------------------------------------------------------
# 8. Metadata
# ------------------------------------------------------------

metadata = {
    "modelo": 4,
    "test": "M4_DESAFIO",
    "proposito": (
        "Instancia desafiante con 3 edificios, 15 salas, 12 cursos por bloque, "
        "flujos divididos, estudiantes entrantes, estudiantes salientes y matriz "
        "completa de costos sala-sala."
    ),
    "status_esperado": "OPTIMAL",
    "costo_esperado": None
}


# ------------------------------------------------------------
# 9. Guardar archivos
# ------------------------------------------------------------

guardar_csv(cursos_b1, "cursos_b1.csv")
guardar_csv(cursos_b2, "cursos_b2.csv")
guardar_csv(salas, "salas.csv")
guardar_csv(flujos, "flujos.csv")
guardar_csv(libres, "libres.csv")
guardar_csv(entrantes, "entrantes.csv")
guardar_csv(costos_sala_sala, "costos_sala_sala.csv")

with open(CARPETA_DESAFIO / "metadata.json", "w", encoding="utf-8") as archivo:
    json.dump(metadata, archivo, ensure_ascii=False, indent=2)

print(f"Instancia desafiante creada en: {CARPETA_DESAFIO}")

Instancia desafiante creada en: c:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\Laboratorio-modelacion-II\data\modelo4\M4_DESAFIO_3_edificios


In [ ]:
from validar_data_4 import validar_instancia_modelo_4
from modelo_4_gurobi import resolver_modelo_4

# Validar estructura de datos
advertencias = validar_instancia_modelo_4(CARPETA_DESAFIO)

print("Validación completada.")

if advertencias:
    print("Advertencias:")
    for adv in advertencias:
        print("-", adv)
else:
    print("No hay advertencias.")


# Resolver con Gurobi
resultado_desafio = resolver_modelo_4(
    CARPETA_DESAFIO,
    mostrar_output=False,
    validar=True,
    time_limit=120
)

print("Status obtenido:", resultado_desafio["status"])
print("Costo objetivo:", resultado_desafio["objetivo"])

In [ ]:
if resultado_desafio["status"] == "OPTIMAL":

    movimientos = resultado_desafio["df_movimientos"].copy()

    mapa_edificios = dict(zip(salas["sala_id"], salas["edificio"]))

    movimientos["edificio_origen"] = movimientos["sala_origen"].map(mapa_edificios)
    movimientos["edificio_destino"] = movimientos["sala_destino"].map(mapa_edificios)

    movimientos["tipo_transicion"] = movimientos.apply(
        lambda fila: (
            "misma sala"
            if fila["sala_origen"] == fila["sala_destino"]
            else "mismo edificio"
            if fila["edificio_origen"] == fila["edificio_destino"]
            else "distinto edificio"
        ),
        axis=1
    )

    flujo_total = movimientos["flujo"].sum()
    costo_total = movimientos["costo_total"].sum()
    flujo_misma_sala = movimientos.loc[
        movimientos["tipo_transicion"] == "misma sala",
        "flujo"
    ].sum()

    resumen_resultado = pd.DataFrame([{
        "status": resultado_desafio["status"],
        "costo_total": costo_total,
        "flujo_total_entre_bloques": flujo_total,
        "flujo_que_permanece_misma_sala": flujo_misma_sala,
        "porcentaje_misma_sala": round(100 * flujo_misma_sala / flujo_total, 2),
        "estudiantes_salientes": libres["libres"].sum(),
        "estudiantes_entrantes": entrantes["entrantes"].sum(),
    }])

    display(Markdown("### Resumen del resultado obtenido"))
    display(resumen_resultado)

    display(Markdown("### Asignación óptima por sala"))
    asignacion = resultado_desafio["df_salas"].copy()

    asignacion = asignacion[
        (asignacion["curso_b1"] != "—") |
        (asignacion["curso_b2"] != "—")
    ]

    display(
        asignacion[
            ["sala", "edificio", "capacidad", "curso_b1", "tamano_b1", "curso_b2", "tamano_b2"]
        ]
    )

    display(Markdown("### Movimientos que generan costo"))
    display(
        movimientos[
            [
                "curso_b1",
                "sala_origen",
                "edificio_origen",
                "curso_b2",
                "sala_destino",
                "edificio_destino",
                "flujo",
                "tipo_transicion",
                "costo_unitario",
                "costo_total"
            ]
        ].sort_values("costo_total", ascending=False)
    )

    display(Markdown("### Costo agrupado por tipo de transición"))
    display(
        movimientos.groupby("tipo_transicion")
        .agg(
            flujo_total=("flujo", "sum"),
            costo_total=("costo_total", "sum"),
            costo_promedio=("costo_unitario", "mean")
        )
        .reset_index()
        .sort_values("costo_total", ascending=False)
    )

else:
    display(Markdown(f"El modelo no obtuvo solución óptima. Status: `{resultado_desafio['status']}`"))